In [ ]:
import { useState, useRef, useCallback } from "react";

// ── Crypto helpers (pure functions, no Web Crypto needed for demo) ──────────
function strToBytes(str) {
  return new TextEncoder().encode(str);
}
function bytesToStr(buf) {
  return new TextDecoder().decode(buf);
}
function toBase64(bytes) {
  let binary = "";
  bytes.forEach((b) => (binary += String.fromCharCode(b)));
  return btoa(binary);
}
function fromBase64(b64) {
  const binary = atob(b64);
  const bytes = new Uint8Array(binary.length);
  for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
  return bytes;
}

// Simple XOR-based key-stretching (for educational demo)
function deriveKey(password, salt) {
  const pw = strToBytes(password);
  const key = new Uint8Array(256);
  for (let i = 0; i < 256; i++) key[i] = pw[i % pw.length] ^ salt[i % salt.length];
  // Two rounds of mixing
  for (let r = 0; r < 2; r++)
    for (let i = 0; i < 256; i++)
      key[i] = ((key[i] ^ key[(i + 1) % 256] ^ r) + i) & 0xff;
  return key;
}

function xorEncrypt(data, key) {
  return data.map((b, i) => b ^ key[i % key.length]);
}

function protect(plaintext, password) {
  const salt = crypto.getRandomValues(new Uint8Array(16));
  const iv   = crypto.getRandomValues(new Uint8Array(8));
  const key  = deriveKey(password, salt);
  const raw  = strToBytes(plaintext);
  // XOR cipher
  const encrypted = xorEncrypt(raw, key.map((k, i) => k ^ iv[i % iv.length]));
  // Simple integrity tag (sum-based)
  let checksum = 0;
  encrypted.forEach((b) => (checksum = (checksum + b) & 0xffff));
  const tag = new Uint8Array([checksum >> 8, checksum & 0xff]);
  // Pack: salt(16) + iv(8) + tag(2) + ciphertext
  const packed = new Uint8Array(16 + 8 + 2 + encrypted.length);
  packed.set(salt, 0);
  packed.set(iv, 16);
  packed.set(tag, 24);
  packed.set(encrypted, 26);
  return "FPU1:" + toBase64(packed);
}

function restore(token, password) {
  if (!token.startsWith("FPU1:")) throw new Error("Invalid protected file format.");
  const packed    = fromBase64(token.slice(5));
  const salt      = packed.slice(0, 16);
  const iv        = packed.slice(16, 24);
  const tag       = packed.slice(24, 26);
  const encrypted = packed.slice(26);
  // Verify checksum
  let checksum = 0;
  encrypted.forEach((b) => (checksum = (checksum + b) & 0xffff));
  if (tag[0] !== (checksum >> 8) || tag[1] !== (checksum & 0xff))
    throw new Error("Integrity check failed. File may be corrupted or tampered.");
  const key = deriveKey(password, salt);
  const decrypted = xorEncrypt(encrypted, key.map((k, i) => k ^ iv[i % iv.length]));
  return bytesToStr(decrypted);
}

// ── UI ───────────────────────────────────────────────────────────────────────
const TABS = ["protect", "restore"];

export default function App() {
  const [tab, setTab]             = useState("protect");
  const [input, setInput]         = useState("");
  const [password, setPassword]   = useState("");
  const [output, setOutput]       = useState("");
  const [status, setStatus]       = useState(null); // {type:"ok"|"err", msg}
  const [showPw, setShowPw]       = useState(false);
  const [strength, setStrength]   = useState(0);
  const [dragging, setDragging]   = useState(false);
  const [fileName, setFileName]   = useState("");
  const fileRef = useRef();

  const calcStrength = (pw) => {
    let s = 0;
    if (pw.length >= 8)  s++;
    if (pw.length >= 14) s++;
    if (/[A-Z]/.test(pw)) s++;
    if (/[0-9]/.test(pw)) s++;
    if (/[^A-Za-z0-9]/.test(pw)) s++;
    return s;
  };

  const handlePwChange = (v) => {
    setPassword(v);
    setStrength(calcStrength(v));
  };

  const loadFile = (file) => {
    setFileName(file.name);
    const reader = new FileReader();
    reader.onload = (e) => setInput(e.target.result);
    reader.readAsText(file);
  };

  const handleDrop = useCallback((e) => {
    e.preventDefault();
    setDragging(false);
    const file = e.dataTransfer.files[0];
    if (file) loadFile(file);
  }, []);

  const run = () => {
    setStatus(null);
    setOutput("");
    if (!input.trim()) return setStatus({ type: "err", msg: "No content to process." });
    if (!password)     return setStatus({ type: "err", msg: "Password is required." });
    try {
      if (tab === "protect") {
        const token = protect(input, password);
        setOutput(token);
        setStatus({ type: "ok", msg: "File protected successfully!" });
      } else {
        const plain = restore(input.trim(), password);
        setOutput(plain);
        setStatus({ type: "ok", msg: "File restored successfully!" });
      }
    } catch (e) {
      setStatus({ type: "err", msg: e.message });
    }
  };

  const download = () => {
    const ext  = tab === "protect" ? ".fpu" : ".txt";
    const mime = tab === "protect" ? "text/plain" : "text/plain";
    const blob = new Blob([output], { type: mime });
    const a    = document.createElement("a");
    a.href     = URL.createObjectURL(blob);
    a.download = (fileName ? fileName.replace(/\.[^.]+$/, "") : "output") + ext;
    a.click();
  };

  const copy = () => {
    navigator.clipboard.writeText(output);
    setStatus({ type: "ok", msg: "Copied to clipboard!" });
  };

  const swStrengthColor = ["#444", "#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#00d4aa"];
  const swStrengthLabel = ["—", "Weak", "Fair", "Moderate", "Strong", "Very Strong"];

  return (
    <div style={S.root}>
      {/* Ambient grid */}
      <div style={S.grid} aria-hidden />

      <div style={S.card}>
        {/* Header */}
        <header style={S.header}>
          <div style={S.lockIcon}>
            <svg width="28" height="28" viewBox="0 0 24 24" fill="none" stroke="#00d4aa" strokeWidth="2" strokeLinecap="round" strokeLinejoin="round">
              <rect x="3" y="11" width="18" height="11" rx="2" ry="2"/>
              <path d="M7 11V7a5 5 0 0 1 10 0v4"/>
            </svg>
          </div>
          <div>
            <h1 style={S.title}>File Protection Utility</h1>
            <p style={S.subtitle}>Encrypt &amp; restore your files with a password</p>
          </div>
        </header>

        {/* Tabs */}
        <div style={S.tabBar}>
          {TABS.map((t) => (
            <button key={t} style={{ ...S.tab, ...(tab === t ? S.tabActive : {}) }}
              onClick={() => { setTab(t); setOutput(""); setStatus(null); }}>
              {t === "protect" ? "🔒 Protect" : "🔓 Restore"}
            </button>
          ))}
        </div>

        {/* Drop zone */}
        <div
          style={{ ...S.dropZone, ...(dragging ? S.dropZoneActive : {}) }}
          onDragOver={(e) => { e.preventDefault(); setDragging(true); }}
          onDragLeave={() => setDragging(false)}
          onDrop={handleDrop}
          onClick={() => fileRef.current.click()}
        >
          <input ref={fileRef} type="file" style={{ display: "none" }}
            accept={tab === "protect" ? ".txt,.csv,.json,.md,.log" : ".fpu,.txt"}
            onChange={(e) => e.target.files[0] && loadFile(e.target.files[0])} />
          <span style={S.dropIcon}>📂</span>
          <span style={S.dropText}>
            {fileName ? `Loaded: ${fileName}` : "Drop a file here or click to upload"}
          </span>
        </div>

        {/* Textarea */}
        <label style={S.label}>
          {tab === "protect" ? "Plain Text Content" : "Protected Token (.fpu)"}
        </label>
        <textarea
          style={S.textarea}
          value={input}
          onChange={(e) => setInput(e.target.value)}
          placeholder={tab === "protect"
            ? "Paste or type the content you want to protect…"
            : "Paste the protected FPU1:… token here…"}
          spellCheck={false}
        />

        {/* Password */}
        <label style={S.label}>Password</label>
        <div style={S.pwRow}>
          <input
            type={showPw ? "text" : "password"}
            style={S.pwInput}
            value={password}
            onChange={(e) => handlePwChange(e.target.value)}
            placeholder="Enter a strong password"
            autoComplete="new-password"
          />
          <button style={S.eyeBtn} onClick={() => setShowPw(!showPw)} title="Toggle visibility">
            {showPw ? "🙈" : "👁️"}
          </button>
        </div>

        {/* Strength bar */}
        {tab === "protect" && password && (
          <div style={S.strengthWrap}>
            <div style={S.strengthBar}>
              {[1,2,3,4,5].map((i) => (
                <div key={i} style={{
                  ...S.strengthSeg,
                  background: i <= strength ? swStrengthColor[strength] : "#2a2a3a"
                }} />
              ))}
            </div>
            <span style={{ ...S.strengthLabel, color: swStrengthColor[strength] }}>
              {swStrengthLabel[strength]}
            </span>
          </div>
        )}

        {/* Action button */}
        <button style={S.runBtn} onClick={run}>
          {tab === "protect" ? "🔐 Encrypt & Protect" : "🔑 Decrypt & Restore"}
        </button>

        {/* Status */}
        {status && (
          <div style={{ ...S.status, ...(status.type === "ok" ? S.statusOk : S.statusErr) }}>
            {status.type === "ok" ? "✅" : "⛔"} {status.msg}
          </div>
        )}

        {/* Output */}
        {output && (
          <div style={S.outputBlock}>
            <div style={S.outputHeader}>
              <span style={S.outputLabel}>
                {tab === "protect" ? "Protected Output" : "Restored Content"}
              </span>
              <div style={S.outputActions}>
                <button style={S.smallBtn} onClick={copy}>Copy</button>
                <button style={S.smallBtn} onClick={download}>Download</button>
              </div>
            </div>
            <pre style={S.outputPre}>{output}</pre>
          </div>
        )}

        {/* Info chips */}
        <div style={S.chips}>
          {["XOR Cipher", "Salt + IV", "Integrity Check", "Base64 Encoding"].map((c) => (
            <span key={c} style={S.chip}>{c}</span>
          ))}
        </div>
      </div>
    </div>
  );
}

// ── Styles ───────────────────────────────────────────────────────────────────
const S = {
  root: {
    minHeight: "100vh",
    background: "#0a0a12",
    display: "flex",
    alignItems: "flex-start",
    justifyContent: "center",
    padding: "40px 16px 60px",
    fontFamily: "'DM Mono', 'Fira Code', 'Courier New', monospace",
    position: "relative",
    overflowX: "hidden",
  },
  grid: {
    position: "fixed",
    inset: 0,
    backgroundImage: `
      linear-gradient(rgba(0,212,170,0.04) 1px, transparent 1px),
      linear-gradient(90deg, rgba(0,212,170,0.04) 1px, transparent 1px)
    `,
    backgroundSize: "40px 40px",
    pointerEvents: "none",
    zIndex: 0,
  },
  card: {
    position: "relative",
    zIndex: 1,
    width: "100%",
    maxWidth: 640,
    background: "linear-gradient(145deg, #12121e 0%, #0e0e1a 100%)",
    border: "1px solid rgba(0,212,170,0.18)",
    borderRadius: 16,
    padding: "36px 32px 32px",
    boxShadow: "0 0 80px rgba(0,212,170,0.07), 0 24px 64px rgba(0,0,0,0.6)",
  },
  header: {
    display: "flex",
    alignItems: "center",
    gap: 14,
    marginBottom: 28,
  },
  lockIcon: {
    width: 52,
    height: 52,
    borderRadius: 14,
    background: "rgba(0,212,170,0.08)",
    border: "1px solid rgba(0,212,170,0.2)",
    display: "flex",
    alignItems: "center",
    justifyContent: "center",
    flexShrink: 0,
  },
  title: {
    margin: 0,
    fontSize: 22,
    fontWeight: 700,
    color: "#e8e8f0",
    letterSpacing: "-0.3px",
  },
  subtitle: {
    margin: "4px 0 0",
    fontSize: 13,
    color: "#666688",
    fontWeight: 400,
  },
  tabBar: {
    display: "flex",
    gap: 8,
    marginBottom: 22,
    background: "#0a0a12",
    borderRadius: 10,
    padding: 4,
    border: "1px solid #1e1e2e",
  },
  tab: {
    flex: 1,
    padding: "9px 0",
    border: "none",
    borderRadius: 7,
    cursor: "pointer",
    fontSize: 13,
    fontFamily: "inherit",
    fontWeight: 600,
    background: "transparent",
    color: "#555577",
    transition: "all 0.2s",
  },
  tabActive: {
    background: "rgba(0,212,170,0.12)",
    color: "#00d4aa",
    boxShadow: "inset 0 0 0 1px rgba(0,212,170,0.25)",
  },
  dropZone: {
    border: "2px dashed #2a2a3e",
    borderRadius: 10,
    padding: "20px 16px",
    display: "flex",
    alignItems: "center",
    gap: 10,
    cursor: "pointer",
    marginBottom: 18,
    transition: "all 0.2s",
    background: "#0d0d18",
  },
  dropZoneActive: {
    borderColor: "#00d4aa",
    background: "rgba(0,212,170,0.05)",
  },
  dropIcon: { fontSize: 20 },
  dropText: { fontSize: 13, color: "#555577" },
  label: {
    display: "block",
    fontSize: 11,
    fontWeight: 700,
    color: "#00d4aa",
    letterSpacing: "0.1em",
    textTransform: "uppercase",
    marginBottom: 6,
  },
  textarea: {
    width: "100%",
    minHeight: 120,
    background: "#0d0d18",
    border: "1px solid #1e1e2e",
    borderRadius: 10,
    padding: "12px 14px",
    color: "#c8c8e0",
    fontFamily: "inherit",
    fontSize: 12,
    resize: "vertical",
    outline: "none",
    marginBottom: 18,
    boxSizing: "border-box",
    lineHeight: 1.6,
    transition: "border-color 0.2s",
  },
  pwRow: {
    display: "flex",
    gap: 8,
    marginBottom: 10,
  },
  pwInput: {
    flex: 1,
    background: "#0d0d18",
    border: "1px solid #1e1e2e",
    borderRadius: 10,
    padding: "10px 14px",
    color: "#c8c8e0",
    fontFamily: "inherit",
    fontSize: 13,
    outline: "none",
    boxSizing: "border-box",
  },
  eyeBtn: {
    width: 42,
    background: "#0d0d18",
    border: "1px solid #1e1e2e",
    borderRadius: 10,
    cursor: "pointer",
    fontSize: 16,
    display: "flex",
    alignItems: "center",
    justifyContent: "center",
    flexShrink: 0,
  },
  strengthWrap: {
    display: "flex",
    alignItems: "center",
    gap: 10,
    marginBottom: 18,
  },
  strengthBar: {
    display: "flex",
    gap: 4,
    flex: 1,
  },
  strengthSeg: {
    flex: 1,
    height: 4,
    borderRadius: 2,
    transition: "background 0.3s",
  },
  strengthLabel: {
    fontSize: 11,
    fontWeight: 700,
    letterSpacing: "0.05em",
    minWidth: 68,
    textAlign: "right",
  },
  runBtn: {
    width: "100%",
    padding: "13px 0",
    background: "linear-gradient(135deg, #00d4aa 0%, #00b894 100%)",
    border: "none",
    borderRadius: 10,
    color: "#0a0a12",
    fontFamily: "inherit",
    fontSize: 14,
    fontWeight: 700,
    cursor: "pointer",
    letterSpacing: "0.03em",
    marginTop: 6,
    marginBottom: 14,
    transition: "opacity 0.2s, transform 0.1s",
    boxShadow: "0 4px 24px rgba(0,212,170,0.25)",
  },
  status: {
    borderRadius: 8,
    padding: "10px 14px",
    fontSize: 13,
    marginBottom: 14,
    fontWeight: 500,
  },
  statusOk: {
    background: "rgba(0,212,170,0.08)",
    border: "1px solid rgba(0,212,170,0.2)",
    color: "#00d4aa",
  },
  statusErr: {
    background: "rgba(231,76,60,0.08)",
    border: "1px solid rgba(231,76,60,0.2)",
    color: "#e74c3c",
  },
  outputBlock: {
    background: "#0d0d18",
    border: "1px solid #1e1e2e",
    borderRadius: 10,
    overflow: "hidden",
    marginBottom: 18,
  },
  outputHeader: {
    display: "flex",
    alignItems: "center",
    justifyContent: "space-between",
    padding: "10px 14px",
    borderBottom: "1px solid #1e1e2e",
    background: "#0a0a12",
  },
  outputLabel: {
    fontSize: 11,
    fontWeight: 700,
    color: "#00d4aa",
    letterSpacing: "0.1em",
    textTransform: "uppercase",
  },
  outputActions: { display: "flex", gap: 6 },
  smallBtn: {
    padding: "5px 12px",
    background: "rgba(0,212,170,0.1)",
    border: "1px solid rgba(0,212,170,0.2)",
    borderRadius: 6,
    color: "#00d4aa",
    fontFamily: "inherit",
    fontSize: 11,
    fontWeight: 600,
    cursor: "pointer",
    letterSpacing: "0.05em",
  },
  outputPre: {
    margin: 0,
    padding: "14px",
    fontSize: 11,
    color: "#888899",
    wordBreak: "break-all",
    whiteSpace: "pre-wrap",
    maxHeight: 180,
    overflowY: "auto",
    lineHeight: 1.6,
  },
  chips: {
    display: "flex",
    flexWrap: "wrap",
    gap: 6,
    marginTop: 4,
  },
  chip: {
    fontSize: 10,
    fontWeight: 600,
    letterSpacing: "0.08em",
    textTransform: "uppercase",
    color: "#444466",
    background: "#0d0d18",
    border: "1px solid #1e1e2e",
    borderRadius: 20,
    padding: "4px 10px",
  },
};
